## 🛠️ 실습 3 - 유사 사례 확장 리서치 (Semantic Search)

<p style="font-size: 16px; font-weight: normal; margin-top: 10px;">
<strong>[미션]</strong> 2장에서 찾은 '수요/판매 예측' 사례를 출발점으로, 유사 사례를 국내외로 확장하라.<br>
이번 실습에서는 <strong>SERPER(키워드 검색)</strong> vs <strong>EXA(의미 기반 검색)</strong>로 동일한 미션을 수행하고 결과를 비교합니다.
</p>

---

## 🛠️ 프로젝트 개발 환경 구축

이 노트북은 **CrewAI**와 **OpenAI**를 활용한 에이전트 개발 환경 설정을 안내합니다.  
아래 단계에 따라 가상환경을 구축하고 필요한 패키지를 설치해 주세요.

### 1. 가상환경 설정
터미널(Terminal)에서(Ctrl+Shift+`) 실행합니다.

* **가상환경 생성:** `env03`이라는 독립된 환경을 구축합니다.
    ```bash
    python -m venv env03
    ```
* **가상환경 활성화:** 생성된 환경으로 진입합니다. (Windows CMD 기준)
    ```bash
    env03\Scripts\activate
    ```

### 2. 필수 라이브러리 설치
진입한 가상환경 터미널에서 아래 명령어를 실행하여 핵심 패키지들을 설치합니다.

| 패키지 | 용도 |
| :--- | :--- |
| **`crewai[tools]`** | CrewAI에 검색·외부 연동 기능을 추가하는 확장 설치 옵션 |
| **`exa-py`** | EXA Search API를 Python에서 사용할 수 있도록 함 |
| **`openai`** | GPT 모델 연동 API |
| **`python-dotenv`** | 환경 변수(.env) 로드 |

```bash
pip install "crewai[tools]" exa-py openai python-dotenv
```

### 1️⃣ 환경 변수 로드

In [14]:
# .env 파일에 아래 키가 정의되어 있어야 합니다.
# - OPENAI_API_KEY
# - OPENAI_MODEL (예: gpt-4o-mini 등)
# - SERPER_API_KEY
# - EXA_API_KEY

import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
SERPER_API_KEY = os.getenv("SERPER_API_KEY")
EXA_API_KEY = os.getenv("EXA_API_KEY")

assert SERPER_API_KEY, "SERPER_API_KEY가 .env에 없습니다."
assert EXA_API_KEY, "EXA_API_KEY가 .env에 없습니다."

print("✅ 환경 변수 로드 완료:", {"OPENAI_MODEL": OPENAI_MODEL})

✅ 환경 변수 로드 완료: {'OPENAI_MODEL': 'gpt-4o-mini'}


### 2️⃣ SERPER Tool 연결

In [15]:
from crewai_tools import SerperDevTool
from crewai import Agent, Task, Crew, LLM

# SERPER_API_KEY를 명시적으로 전달하여 어떤 키가 사용되는지 명확히 합니다.
serper_tool = SerperDevTool(api_key=SERPER_API_KEY)
print("✅ SERPER Tool 연결 완료")

✅ SERPER Tool 연결 완료


### 3️⃣ EXA 연결 및 Custom Tool 생성

In [16]:
from exa_py import Exa # EXA API 클라이언트 라이브러리
from crewai.tools import tool # 툴 데코레이터 임포트

exa = Exa(api_key=EXA_API_KEY) # EXA API 키를 명시적으로 전달하여 어떤 키가 사용되는지 명확히 합니다.

@tool("exa_semantic_search") # 이 함수는 이제 exa_semantic_search라는 이름의 CrewAI 전용 도구가 됩니다.
def exa_semantic_search(query: str) -> str:
    # LLM 에이전트는 이 주석 내용을 읽고 "아, 내가 수요/판매 예측 유사 사례를 확장하고 싶을 때 이 툴을 꺼내 써야겠구나!" 하고 
    # 판단하게 됩니다. 즉, 에이전트를 위한 '툴 사용 설명서'입니다.
    """의미 기반 검색(semantic search)으로 유사 사례를 확장합니다. 

    - 입력 query(한글 가능)를 바탕으로,
      과거 상품/판매 데이터와 고객 데이터를 활용한 수요/판매량 예측 사례를
      국내/해외에서 폭넓게 찾도록 검색을 보강합니다.
    - 반환값은 사람이 읽을 수 있는 요약 문자열입니다.
    """
    # 에이전트가 입력한 원본 query를 그대로 검색하지 않고, 영어 기반의 정교한 리서치 요청 프롬프트로 감싸서(Wrapping) 
    # Exa 검색 엔진에 전달합니다.
    improved_query = f"""
    Find real-world case studies where companies used historical product/sales data AND customer data
    to forecast future demand and/or sales volume.
    Include implementation details, business context, and measurable outcomes.
    Prefer credible sources (company blogs, vendor case studies, reputable media, research reports).

    User query: {query}
    """.strip()

    # 확장된 쿼리를 바탕으로 의미 기반 검색을 수행합니다.
    results = exa.search( 
        improved_query,
        num_results=6, # 최대 6개의 유사 사례를 반환하도록 설정
        contents={"text": True} # 검색 결과에서 텍스트 콘텐츠도 포함하도록 요청
    )

    # Exa 검색 엔진이 반환한 raw 데이터를 에이전트(LLM)가 읽기 좋은 깔끔한 텍스트 구조로 정제하는 루프
    lines = []
    for i, r in enumerate(results.results, start=1):
        title = getattr(r, "title", "") or "(no title)"
        url = getattr(r, "url", "") or ""
        text = (getattr(r, "text", "") or "").strip().replace("\n", " ")
        snippet = (text[:280] + "…") if len(text) > 280 else text
        lines.append(f"[{i}] {title}\n- URL: {url}\n- Snippet: {snippet}\n")

    return "\n".join(lines)

print("✅ EXA Custom Tool 준비 완료")

✅ EXA Custom Tool 준비 완료


### 4️⃣ 리서치 기준(Seed) 설정

> 2장에서 찾은 AI 기반 데이터 분석 자동화 사례 중,  
> **'과거 상품/판매 데이터와 고객 데이터를 결합해 수요/판매량을 예측'** 한 사례를 출발점으로 삼습니다.

- 예: *"과거 상품 판매 데이터 + 고객 구매/행동 데이터를 활용해 수요/판매량을 예측"*

In [17]:
# 2장에서 발견한 사례를 출발점(Seed)으로 설정합니다.
# 아래 seed_case는 '유사성(semantic similarity)' 판단 기준이 됩니다.

seed_case = "과거 상품/판매 데이터와 고객 데이터를 결합해 미래 수요와 판매량을 예측(수요예측/판매예측)한 기업 사례"

print("✅ Seed Case:", seed_case)

✅ Seed Case: 과거 상품/판매 데이터와 고객 데이터를 결합해 미래 수요와 판매량을 예측(수요예측/판매예측)한 기업 사례


### 5️⃣ Agent 정의 (SERPER / EXA)

In [18]:
llm = LLM(model=OPENAI_MODEL)

research_agent_serper = Agent(
    role="Research Analyst (Keyword Search)",
    goal="Seed Case와 유사한 '수요/판매 예측' 사례를 키워드 검색으로 확장한다.",
    backstory="당신은 기업 사례 리서치 전문가이며, 검색 결과를 근거 중심으로 정리합니다. 모든 결과물은 반드시 한국어로 작성합니다.",
    tools=[serper_tool],
    llm=llm,
    max_iter=5,  # Tool 호출 횟수 제한
    verbose=True
)

research_agent_exa = Agent(
    role="Research Analyst (Semantic Search)",
    goal="Seed Case와 유사한 '수요/판매 예측' 사례를 의미 기반 검색으로 확장한다.",
    backstory="당신은 의미 기반 리서치 전문가이며, 서로 다른 표현을 가진 유사 사례를 확장합니다. 모든 결과물은 반드시 한국어로 작성합니다.",
    tools=[exa_semantic_search],
    llm=llm,
    max_iter=5,  # Tool 호출 횟수 제한
    verbose=True
)

print("✅ Agent 정의 완료")

✅ Agent 정의 완료


### 6️⃣ Task 정의

In [19]:
task_description = f"""
[미션]
2장에서 찾은 기업 사례(Seed)를 출발점으로, 유사 사례를 국내외로 확장하라.

[Seed Case]
{seed_case}

[요구사항]
- 국내 1개 이상 + 해외 2개 이상 포함 (총 4~6개)
- '과거 상품/판매 데이터'와 '고객 데이터'가 예측에 활용된 사례를 우선
- 각 사례는 아래 항목을 포함해 정리
  1) 기업/조직명
  2) 국가(국내/해외)
  3) 산업
  4) 도입 배경
  5) 실행 방식
  6) 성과/효과
  7) 출처 URL
  8) Seed Case와의 유사점(한 줄)

[주의]
- 출처 URL은 반드시 포함
- 광고성/홍보성 문서만으로 구성되지 않도록 다양한 출처를 섞어라
- 반드시 한국어로 작성하라
""".strip()

research_task_serper = Task(
    description=task_description,
    expected_output="유사 사례 4~6개를 구조화해 정리(국내 1+ 해외 2+ 포함).",
    agent=research_agent_serper
)

research_task_exa = Task(
    description=task_description,
    expected_output="유사 사례 4~6개를 구조화해 정리(국내 1+ 해외 2+ 포함).",
    agent=research_agent_exa
)

print("✅ Task 정의 완료")

✅ Task 정의 완료


### 7️⃣ Crew 구성 (SERPER)

In [20]:
crew_serper = Crew(
    agents=[research_agent_serper],
    tasks=[research_task_serper],
    verbose=True
)

### 8️⃣ SERPER 실행과 최종 결과 출력

In [21]:
# 기존: serper_result = crew_serper.kickoff()
# 🛠️ 비동기 방식으로 수정:
serper_result = await crew_serper.kickoff_async()

print("\n===== SERPER 결과 =====\n")
print(serper_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6b52bb18-9798-4e70-9346-95b10244c99f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst (Keyword Search)                                                                       │
│                                                                                                                 │
│  Task: [미션]                                                                                                   │
│  2장에서 찾은 기업 사례(Seed)를 출발점으로, 유사 사례를 국내외로 확장하라.                                      │
│                                                                                                                 │
│  [Seed Case]                                                                                                    │
│  과거 상품/판매 데이터와 고객 데이터를 결합해 미래 수요와 판매량을 예측(수요예측/판매예측)한 기업 사례          │
│                                                                                                                 │
│  [요구사항]                                                                                                     │
│  - 국내 1개 이상 + 해외 2개 이상 포함 (총 4~6개)                                                                │
│  - '과거 상품/판매 데이터'와 '고객 데이터'가 예측에 활용된 사례를 우선                                          │
│  - 각 사례는 아래 항목을 포함해 정리                                                                            │
│    1) 기업/조직명                                                                                               │
│    2) 국가(국내/해외)                                                                                           │
│    3) 산업                                                                                                      │
│    4) 도입 배경                                                                                                 │
│    5) 실행 방식                                                                                                 │
│    6) 성과/효과                                                                                                 │
│    7) 출처 URL                                                                                                  │
│    8) Seed Case와의 유사점(한 줄)                                                                               │
│                                                                                                                 │
│  [주의]                                                                                                         │
│  - 출처 URL은 반드시 포함                                                                                       │
│  - 광고성/홍보성 문서만으로 구성되지 않도록 다양한 출처를 섞어라                                                │
│  - 반드시 한국어로 작성하라                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: [미션]                                                                                                   │
│  2장에서 찾은 기업 사례(Seed)를 출발점으로, 유사 사례를 국내외로 확장하라.                                      │
│                                                                                                                 │
│  [Seed Case]                                                                                                    │
│  과거 상품/판매 데이터와 고객 데이터를 결합해 미래 수요와 판매량을 예측(수요예측/판매예측)한 기업 사례          │
│                                                                                                                 │
│  [요구사항]                                                                                                     │
│  - 국내 1개 이상 + 해외 2개 이상 포함 (총 4~6개)                                                                │
│  - '과거 상품/판매 데이터'와 '고객 데이터'가 예측에 활용된 사례를 우선                                          │
│  - 각 사례는 아래 항목을 포함해 정리                                                                            │
│    1) 기업/조직명                                                                                               │
│    2) 국가(국내/해외)                                                                                           │
│    3) 산업                                                                                                      │
│    4) 도입 배경                                                                                                 │
│    5) 실행 방식                                                                                                 │
│    6) 성과/효과                                                                                                 │
│    7) 출처 URL                                                                                                  │
│    8) Seed Case와의 유사점(한 줄)                                                                               │
│                                                                                                                 │
│  [주의]                                                                                                         │
│  - 출처 URL은 반드시 포함                                                                                       │
│  - 광고성/홍보성 문서만으로 구성되지 않도록 다양한 출처를 섞어라                                                │
│  - 반드시 한국어로 작성하라                                                                                     │
│  ID: 356d06fe-8617-4925-ba8e-91076f6b9f5d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': '과거 상품 판매 데이터 고객 데이터 수요 예측 사례 국내'}                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#5) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': '과거 상품 판매 데이터 고객 데이터 수요 예측 사례 해외'}                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst (Keyword Search)                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  다음은 유사한 '수요/판매 예측' 사례를 포함한 정리입니다. Seed Case와 비교하여 각 사례를 구조화하여             │
│  요약하였습니다.                                                                                                │
│                                                                                                                 │
│  ### 1. 이마트                                                                                                  │
│  1) **기업/조직명**: 이마트                                                                                     │
│  2) **국가**: 대한민국                                                                                          │
│  3) **산업**: 유통                                                                                              │
│  4) **도입 배경**: 고객 수요를 정확히 예측하여 재고 비용을 최소화하고, 판매 효율을 높이기 위해 AI 기반의 수요   │
│  예측 시스템 도입.                                                                                              │
│  5) **실행 방식**: 이마트는 자사 데이터를 바탕으로 고급 분석 알고리즘을 활용하여 판매 예측을 실시하고, 신제품   │
│  출시 시 사전 수요 예측을 통해 판매량을 조절.                                                                   │
│  6) **성과/효과**: 재고 관리 최적화와 판매량 증가를 통해 운영 비용 절감 및 고객만족도 향상.                     │
│  7) **출처 URL**: [브런치 - 기업 생존을 좌우하는 수요예측](https://brunch.co.kr/@vagabondboy/119)               │
│  8) **Seed Case와의 유사점**: 과거 데이터를 기반으로 AI를 통해 수요 예측을 진행.                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. 아마존                                                                                                  │
│  1) **기업/조직명**: 아마존                                                                                     │
│  2) **국가**: 미국                                                                                              │
│  3) **산업**: 전자상거래                                                                                        │
│  4) **도입 배경**: 상품 수요 예측의 정확도를 높이고, 재고 관리를 최적화하여 고객 만족도를 높이기 위해 도입.     │
│  5) **실행 방식**: 고객의 과거 구매 이력을 분석하여 AI 알고리즘을 이용해 미래의 판매량과 트렌드를 예측, 이를    │
│  통해 프로모션 및 재고 문제를 해결.                                                                             │
│  6) **성과/효과**: 판매 예측 정확도가 8% 향상되었고, 연간 상당한 비용 절감 효과 (약 55만 3,000 USD)를 기록.     │
│  7) **출처 URL**: [Amazon Forecast 고객](https://aws.amazon.com/ko/forecast/customers/)                         │
│  8) **Seed Case와의 유사점**: 과거의 고객 데이터와 판매 데이터를 결합하여 미래 수요를 예측.                     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 3. Walmart (월마트)                                                                                        │
│  1) **기업/조직명**: Walmart                                                                                    │
│  2) **국가**: 미국                                                                                              │
│  3) **산업**: 유통                          

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: [미션]                                                                                                   │
│  2장에서 찾은 기업 사례(Seed)를 출발점으로, 유사 사례를 국내외로 확장하라.                                      │
│                                                                                                                 │
│  [Seed Case]                                                                                                    │
│  과거 상품/판매 데이터와 고객 데이터를 결합해 미래 수요와 판매량을 예측(수요예측/판매예측)한 기업 사례          │
│                                                                                                                 │
│  [요구사항]                                                                                                     │
│  - 국내 1개 이상 + 해외 2개 이상 포함 (총 4~6개)                                                                │
│  - '과거 상품/판매 데이터'와 '고객 데이터'가 예측에 활용된 사례를 우선                                          │
│  - 각 사례는 아래 항목을 포함해 정리                                                                            │
│    1) 기업/조직명                                                                                               │
│    2) 국가(국내/해외)                                                                                           │
│    3) 산업                                                                                                      │
│    4) 도입 배경                                                                                                 │
│    5) 실행 방식                                                                                                 │
│    6) 성과/효과                                                                                                 │
│    7) 출처 URL                                                                                                  │
│    8) Seed Case와의 유사점(한 줄)                                                                               │
│                                                                                                                 │
│  [주의]                                                                                                         │
│  - 출처 URL은 반드시 포함                                                                                       │
│  - 광고성/홍보성 문서만으로 구성되지 않도록 다양한 출처를 섞어라                                                │
│  - 반드시 한국어로 작성하라                                                                                     │
│  Agent: Research Analyst (Keyword Search)                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== SERPER 결과 =====

다음은 유사한 '수요/판매 예측' 사례를 포함한 정리입니다. Seed Case와 비교하여 각 사례를 구조화하여 요약하였습니다.

### 1. 이마트
1) **기업/조직명**: 이마트  
2) **국가**: 대한민국  
3) **산업**: 유통  
4) **도입 배경**: 고객 수요를 정확히 예측하여 재고 비용을 최소화하고, 판매 효율을 높이기 위해 AI 기반의 수요 예측 시스템 도입.  
5) **실행 방식**: 이마트는 자사 데이터를 바탕으로 고급 분석 알고리즘을 활용하여 판매 예측을 실시하고, 신제품 출시 시 사전 수요 예측을 통해 판매량을 조절.  
6) **성과/효과**: 재고 관리 최적화와 판매량 증가를 통해 운영 비용 절감 및 고객만족도 향상.  
7) **출처 URL**: [브런치 - 기업 생존을 좌우하는 수요예측](https://brunch.co.kr/@vagabondboy/119)  
8) **Seed Case와의 유사점**: 과거 데이터를 기반으로 AI를 통해 수요 예측을 진행.

---

### 2. 아마존
1) **기업/조직명**: 아마존  
2) **국가**: 미국  
3) **산업**: 전자상거래  
4) **도입 배경**: 상품 수요 예측의 정확도를 높이고, 재고 관리를 최적화하여 고객 만족도를 높이기 위해 도입.  
5) **실행 방식**: 고객의 과거 구매 이력을 분석하여 AI 알고리즘을 이용해 미래의 판매량과 트렌드를 예측, 이를 통해 프로모션 및 재고 문제를 해결.  
6) **성과/효과**: 판매 예측 정확도가 8% 향상되었고, 연간 상당한 비용 절감 효과 (약 55만 3,000 USD)를 기록.  
7) **출처 URL**: [Amazon Forecast 고객](https://aws.amazon.com/ko/forecast/customers/)  
8) **Seed Case와의 유사점**: 과거의 고객 데이터와 판매 데이터를 결합하여 미래 수요를 예측.

---

### 3. Wa

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6b52bb18-9798-4e70-9346-95b10244c99f                                                                       │
│  Final Output: 다음은 유사한 '수요/판매 예측' 사례를 포함한 정리입니다. Seed Case와 비교하여 각 사례를          │
│  구조화하여 요약하였습니다.                                                                                     │
│                                                                                                                 │
│  ### 1. 이마트                                                                                                  │
│  1) **기업/조직명**: 이마트                                                                                     │
│  2) **국가**: 대한민국                                                                                          │
│  3) **산업**: 유통                                                                                              │
│  4) **도입 배경**: 고객 수요를 정확히 예측하여 재고 비용을 최소화하고, 판매 효율을 높이기 위해 AI 기반의 수요   │
│  예측 시스템 도입.                                                                                              │
│  5) **실행 방식**: 이마트는 자사 데이터를 바탕으로 고급 분석 알고리즘을 활용하여 판매 예측을 실시하고, 신제품   │
│  출시 시 사전 수요 예측을 통해 판매량을 조절.                                                                   │
│  6) **성과/효과**: 재고 관리 최적화와 판매량 증가를 통해 운영 비용 절감 및 고객만족도 향상.                     │
│  7) **출처 URL**: [브런치 - 기업 생존을 좌우하는 수요예측](https://brunch.co.kr/@vagabondboy/119)               │
│  8) **Seed Case와의 유사점**: 과거 데이터를 기반으로 AI를 통해 수요 예측을 진행.                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. 아마존                                                                                                  │
│  1) **기업/조직명**: 아마존                                                                                     │
│  2) **국가**: 미국                                                                                              │
│  3) **산업**: 전자상거래                                                                                        │
│  4) **도입 배경**: 상품 수요 예측의 정확도를 높이고, 재고 관리를 최적화하여 고객 만족도를 높이기 위해 도입.     │
│  5) **실행 방식**: 고객의 과거 구매 이력을 분석하여 AI 알고리즘을 이용해 미래의 판매량과 트렌드를 예측, 이를    │
│  통해 프로모션 및 재고 문제를 해결.                                                                             │
│  6) **성과/효과**: 판매 예측 정확도가 8% 향상되었고, 연간 상당한 비용 절감 효과 (약 55만 3,000 USD)를 기록.     │
│  7) **출처 URL**: [Amazon Forecast 고객](https://aws.amazon.com/ko/forecast/customers/)                         │
│  8) **Seed Case와의 유사점**: 과거의 고객 데이터와 판매 데이터를 결합하여 미래 수요를 예측.                     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 3. Walmart (월마트)                                                                                        │
│  1) **기업/조직명**: Walmart                                                                                    │
│  2) **국가**: 미국                                                                                              │
│  3) **산업**: 유통                         

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 9️⃣ Crew 구성 (EXA)

In [22]:
crew_exa = Crew(
    agents=[research_agent_exa],
    tasks=[research_task_exa],
    verbose=True
)

### 🔟 EXA 실행과 최종 결과 출력

In [23]:
# 기존: exa_result = crew_exa.kickoff()
# 🛠️ 비동기 방식으로 수정:
exa_result = await crew_exa.kickoff_async()

print("\n===== EXA 결과 =====\n")
print(exa_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ec82f89b-122e-45e6-97d2-a128e8be5854                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: [미션]                                                                                                   │
│  2장에서 찾은 기업 사례(Seed)를 출발점으로, 유사 사례를 국내외로 확장하라.                                      │
│                                                                                                                 │
│  [Seed Case]                                                                                                    │
│  과거 상품/판매 데이터와 고객 데이터를 결합해 미래 수요와 판매량을 예측(수요예측/판매예측)한 기업 사례          │
│                                                                                                                 │
│  [요구사항]                                                                                                     │
│  - 국내 1개 이상 + 해외 2개 이상 포함 (총 4~6개)                                                                │
│  - '과거 상품/판매 데이터'와 '고객 데이터'가 예측에 활용된 사례를 우선                                          │
│  - 각 사례는 아래 항목을 포함해 정리                                                                            │
│    1) 기업/조직명                                                                                               │
│    2) 국가(국내/해외)                                                                                           │
│    3) 산업                                                                                                      │
│    4) 도입 배경                                                                                                 │
│    5) 실행 방식                                                                                                 │
│    6) 성과/효과                                                                                                 │
│    7) 출처 URL                                                                                                  │
│    8) Seed Case와의 유사점(한 줄)                                                                               │
│                                                                                                                 │
│  [주의]                                                                                                         │
│  - 출처 URL은 반드시 포함                                                                                       │
│  - 광고성/홍보성 문서만으로 구성되지 않도록 다양한 출처를 섞어라                                                │
│  - 반드시 한국어로 작성하라                                                                                     │
│  ID: c87c2243-d232-42c3-8df6-c7423e14a016                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst (Semantic Search)                                                                      │
│                                                                                                                 │
│  Task: [미션]                                                                                                   │
│  2장에서 찾은 기업 사례(Seed)를 출발점으로, 유사 사례를 국내외로 확장하라.                                      │
│                                                                                                                 │
│  [Seed Case]                                                                                                    │
│  과거 상품/판매 데이터와 고객 데이터를 결합해 미래 수요와 판매량을 예측(수요예측/판매예측)한 기업 사례          │
│                                                                                                                 │
│  [요구사항]                                                                                                     │
│  - 국내 1개 이상 + 해외 2개 이상 포함 (총 4~6개)                                                                │
│  - '과거 상품/판매 데이터'와 '고객 데이터'가 예측에 활용된 사례를 우선                                          │
│  - 각 사례는 아래 항목을 포함해 정리                                                                            │
│    1) 기업/조직명                                                                                               │
│    2) 국가(국내/해외)                                                                                           │
│    3) 산업                                                                                                      │
│    4) 도입 배경                                                                                                 │
│    5) 실행 방식                                                                                                 │
│    6) 성과/효과                                                                                                 │
│    7) 출처 URL                                                                                                  │
│    8) Seed Case와의 유사점(한 줄)                                                                               │
│                                                                                                                 │
│  [주의]                                                                                                         │
│  - 출처 URL은 반드시 포함                                                                                       │
│  - 광고성/홍보성 문서만으로 구성되지 않도록 다양한 출처를 섞어라                                                │
│  - 반드시 한국어로 작성하라                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: exa_semantic_search                                                                                      │
│  Args: {'query': '과거 상품 판매 데이터와 고객 데이터를 활용한 수요 예측 사례'}                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#5) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: exa_semantic_search                                                                                      │
│  Args: {'query': '판매량 예측 기업 사례'}                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research Analyst (Semantic Search)                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  다음은 과거 상품/판매 데이터와 고객 데이터를 활용하여 수요 및 판매량 예측을 성공적으로 수행한 국내외           │
│  사례들입니다.                                                                                                  │
│                                                                                                                 │
│  ### 1. 일동후디스                                                                                              │
│  1) **기업/조직명**: 일동후디스                                                                                 │
│  2) **국가**: 국내                                                                                              │
│  3) **산업**: 식품                                                                                              │
│  4) **도입 배경**: 재고 리스크를 줄이고 수요 예측의 정확성을 높이기 위하여 AI 기술 도입 필요성 대두             │
│  5) **실행 방식**: 임팩티브AI의 '딥플로우' 솔루션을 사용하여 AI 기반 수요 예측 모델을 구축                      │
│  6) **성과/효과**: 재고 리스크 26% 감소 및 운영 효율성 향상                                                     │
│  7) **출처 URL**:                                                                                               │
│  [임팩티브AI](https://www.impactive-ai.com/clients/ildong-foodis-reduces-inventory-risk-by-26-with-ai-demand-f  │
│  orecasting)                                                                                                    │
│  8) **Seed Case와의 유사점**: 과거 판매 데이터를 기반으로 수요 예측을 통해 재고 관리를 최적화함.                │
│                                                                                                                 │
│  ### 2. 한미사이언스                                                                                            │
│  1) **기업/조직명**: 한미사이언스                                                                               │
│  2) **국가**: 국내                                                                                              │
│  3) **산업**: 제약                                                                                              │
│  4) **도입 배경**: 효율적인 수요 관리와 재고 비용 절감을 위해 AI 도입                                           │
│  5) **실행 방식**: 임팩티브AI의 딥플로우 솔루션을 통해 고객 데이터와 판매 데이터를 분석하여 수요 예측           │
│  6) **성과/효과**: 재고비용 절감 및 업무 효율화 이루어짐                                                        │
│  7) **출처 URL**: [포인트경제](https://www.pointe.co.kr/news/articleView.html?idxno=34168)                      │
│  8) **Seed Case와의 유사점**: 고객 데이터와 판매 데이터를 결합하여 수요 예측 시스템을 구축함.                   │
│                                                                                                                 │
│  ### 3. CJ대한통운                                                                                              │
│  1) **기업/조직명**: CJ대한통운                                                                                 │
│  2) **국가**: 국내                                                                                              │
│  3) **산업**: 물류                                                                                              │
│  4) **도입 배경**: 이커머스 성장에 따른 주문량 예측의 중요성 증가                                               │
│  5) **실행 방식**: AI 및 빅데이터 기술을 활용하여 고객 행동 분석과 주문 패턴을 기반으로 수요 예측 시스템 구축   │
│  6) **성과/효과**: 평균 예측 정확도 88% 달성, 고객 선호 트렌드 정보 제공                                        │
│  7) **출처 URL**: [해양한국](https://www.monthlymaritimekorea.com/news/articleView.html?idxno=32899)            │
│  8) **Seed Case와의 유사점**: 판매 데이터와 고객 데이터를 활용하여 수요 예측의 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: [미션]                                                                                                   │
│  2장에서 찾은 기업 사례(Seed)를 출발점으로, 유사 사례를 국내외로 확장하라.                                      │
│                                                                                                                 │
│  [Seed Case]                                                                                                    │
│  과거 상품/판매 데이터와 고객 데이터를 결합해 미래 수요와 판매량을 예측(수요예측/판매예측)한 기업 사례          │
│                                                                                                                 │
│  [요구사항]                                                                                                     │
│  - 국내 1개 이상 + 해외 2개 이상 포함 (총 4~6개)                                                                │
│  - '과거 상품/판매 데이터'와 '고객 데이터'가 예측에 활용된 사례를 우선                                          │
│  - 각 사례는 아래 항목을 포함해 정리                                                                            │
│    1) 기업/조직명                                                                                               │
│    2) 국가(국내/해외)                                                                                           │
│    3) 산업                                                                                                      │
│    4) 도입 배경                                                                                                 │
│    5) 실행 방식                                                                                                 │
│    6) 성과/효과                                                                                                 │
│    7) 출처 URL                                                                                                  │
│    8) Seed Case와의 유사점(한 줄)                                                                               │
│                                                                                                                 │
│  [주의]                                                                                                         │
│  - 출처 URL은 반드시 포함                                                                                       │
│  - 광고성/홍보성 문서만으로 구성되지 않도록 다양한 출처를 섞어라                                                │
│  - 반드시 한국어로 작성하라                                                                                     │
│  Agent: Research Analyst (Semantic Search)                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


===== EXA 결과 =====

다음은 과거 상품/판매 데이터와 고객 데이터를 활용하여 수요 및 판매량 예측을 성공적으로 수행한 국내외 사례들입니다.

### 1. 일동후디스
1) **기업/조직명**: 일동후디스  
2) **국가**: 국내  
3) **산업**: 식품  
4) **도입 배경**: 재고 리스크를 줄이고 수요 예측의 정확성을 높이기 위하여 AI 기술 도입 필요성 대두  
5) **실행 방식**: 임팩티브AI의 '딥플로우' 솔루션을 사용하여 AI 기반 수요 예측 모델을 구축   
6) **성과/효과**: 재고 리스크 26% 감소 및 운영 효율성 향상  
7) **출처 URL**: [임팩티브AI](https://www.impactive-ai.com/clients/ildong-foodis-reduces-inventory-risk-by-26-with-ai-demand-forecasting)  
8) **Seed Case와의 유사점**: 과거 판매 데이터를 기반으로 수요 예측을 통해 재고 관리를 최적화함.

### 2. 한미사이언스
1) **기업/조직명**: 한미사이언스  
2) **국가**: 국내  
3) **산업**: 제약  
4) **도입 배경**: 효율적인 수요 관리와 재고 비용 절감을 위해 AI 도입  
5) **실행 방식**: 임팩티브AI의 딥플로우 솔루션을 통해 고객 데이터와 판매 데이터를 분석하여 수요 예측  
6) **성과/효과**: 재고비용 절감 및 업무 효율화 이루어짐  
7) **출처 URL**: [포인트경제](https://www.pointe.co.kr/news/articleView.html?idxno=34168)  
8) **Seed Case와의 유사점**: 고객 데이터와 판매 데이터를 결합하여 수요 예측 시스템을 구축함.

### 3. CJ대한통운
1) **기업/조직명**: CJ대한통운  
2) **국가**: 국내  
3) **산업**: 물류  
4) **도입 배경**: 이커머스 성장에 따른 주문량 예측의 중요성 증가  


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ec82f89b-122e-45e6-97d2-a128e8be5854                                                                       │
│  Final Output: 다음은 과거 상품/판매 데이터와 고객 데이터를 활용하여 수요 및 판매량 예측을 성공적으로 수행한    │
│  국내외 사례들입니다.                                                                                           │
│                                                                                                                 │
│  ### 1. 일동후디스                                                                                              │
│  1) **기업/조직명**: 일동후디스                                                                                 │
│  2) **국가**: 국내                                                                                              │
│  3) **산업**: 식품                                                                                              │
│  4) **도입 배경**: 재고 리스크를 줄이고 수요 예측의 정확성을 높이기 위하여 AI 기술 도입 필요성 대두             │
│  5) **실행 방식**: 임팩티브AI의 '딥플로우' 솔루션을 사용하여 AI 기반 수요 예측 모델을 구축                      │
│  6) **성과/효과**: 재고 리스크 26% 감소 및 운영 효율성 향상                                                     │
│  7) **출처 URL**:                                                                                               │
│  [임팩티브AI](https://www.impactive-ai.com/clients/ildong-foodis-reduces-inventory-risk-by-26-with-ai-demand-f  │
│  orecasting)                                                                                                    │
│  8) **Seed Case와의 유사점**: 과거 판매 데이터를 기반으로 수요 예측을 통해 재고 관리를 최적화함.                │
│                                                                                                                 │
│  ### 2. 한미사이언스                                                                                            │
│  1) **기업/조직명**: 한미사이언스                                                                               │
│  2) **국가**: 국내                                                                                              │
│  3) **산업**: 제약                                                                                              │
│  4) **도입 배경**: 효율적인 수요 관리와 재고 비용 절감을 위해 AI 도입                                           │
│  5) **실행 방식**: 임팩티브AI의 딥플로우 솔루션을 통해 고객 데이터와 판매 데이터를 분석하여 수요 예측           │
│  6) **성과/효과**: 재고비용 절감 및 업무 효율화 이루어짐                                                        │
│  7) **출처 URL**: [포인트경제](https://www.pointe.co.kr/news/articleView.html?idxno=34168)                      │
│  8) **Seed Case와의 유사점**: 고객 데이터와 판매 데이터를 결합하여 수요 예측 시스템을 구축함.                   │
│                                                                                                                 │
│  ### 3. CJ대한통운                                                                                              │
│  1) **기업/조직명**: CJ대한통운                                                                                 │
│  2) **국가**: 국내                                                                                              │
│  3) **산업**: 물류                                                                                              │
│  4) **도입 배경**: 이커머스 성장에 따른 주문량 예측의 중요성 증가                                               │
│  5) **실행 방식**: AI 및 빅데이터 기술을 활용하여 고객 행동 분석과 주문 패턴을 기반으로 수요 예측 시스템 구축   │
│  6) **성과/효과**: 평균 예측 정확도 88% 달성, 고객 선호 트렌드 정보 제공                                        │
│  7) **출처 URL**: [해양한국](https://www.monthlymaritimekorea.com/news/articleView.html?idxno=32899)            │
│  8) **Seed Case와의 유사점**: 판매 데이터와 고객 데이터를 활용하여 수요 예측의

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 1️⃣1️⃣ 정량 비교

In [24]:
import re

def simple_stats(text: str) -> dict:
    t = str(text)
    urls = re.findall(r'https?://\S+', t)
    lines = [ln for ln in t.splitlines() if ln.strip()]
    return {
        "총 문자 수": len(t),
        "총 라인 수": len(lines),
        "URL 개수": len(urls),
        "고유 URL 개수": len(set(urls)),
    }

print("SERPER 통계:", simple_stats(serper_result))
print("EXA    통계:", simple_stats(exa_result))

SERPER 통계: {'총 문자 수': 1864, '총 라인 수': 41, 'URL 개수': 4, '고유 URL 개수': 4}
EXA    통계: {'총 문자 수': 1709, '총 라인 수': 38, 'URL 개수': 4, '고유 URL 개수': 4}


### 1️⃣2️⃣ LLM as a Judge (유사 사례 확장 품질 평가)

> LLM as a Judge는 평가 목적이므로, CrewAI Agent 없이 OpenAI API를 직접 호출합니다.  
> Agent의 Tool 호출 오버헤드 없이 단순 평가 프롬프트만 실행하면 충분하기 때문입니다.

In [25]:
from openai import OpenAI

client = OpenAI()

evaluation_prompt = f"""
당신은 '유사 사례 확장 리서치' 품질을 평가하는 심사위원이다.
아래는 동일한 미션(Seed Case 기반 유사 사례 확장)에 대해 두 검색 도구가 생성한 결과이다.

[미션]
2장에서 찾은 기업 사례(Seed)를 출발점으로, 유사 사례를 국내외로 확장하라.
Seed Case: {seed_case}

[ SERPER (Keyword Search) 결과 ]
{serper_result}

[ EXA (Semantic Search) 결과 ]
{exa_result}

다음 기준으로 각각 1~5점으로 채점하고, 근거를 2~3문장으로 설명하라.

1) 유사성 적합도: Seed Case와 '문제/데이터/목표' 관점에서 얼마나 유사한가?
2) 확장성: 표현이 달라져도 새로운 산업/맥락으로 자연스럽게 확장되는가?
3) 근거 품질: 출처 URL의 신뢰도/다양성, 구체적 디테일의 수준
4) 중복 최소화: 서로 비슷한 사례 반복이 적고, 사례 간 구분이 명확한가?

마지막에 총점(20점 만점)과 승자를 선언하라.
단, '키워드 검색이 항상 나쁘다' 같은 일반화는 하지 말고, 이번 미션에 한정해 판단하라.
""".strip()

resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {"role": "system", "content": "You are a rigorous evaluator. Respond in Korean. Be fair and concise."},
        {"role": "user", "content": evaluation_prompt},
    ],
    temperature=0.2,  # 낮은 온도로 일관된 평가 유도
)

print("\n===== LLM 평가 결과 (유사 사례 확장 중심) =====\n")
print(resp.choices[0].message.content)


===== LLM 평가 결과 (유사 사례 확장 중심) =====

### SERPER (Keyword Search) 평가

1) **유사성 적합도**: 4점  
   - Seed Case와 각 사례는 모두 과거 데이터와 고객 데이터를 활용하여 수요 및 판매 예측을 수행하는 점에서 유사하다. 그러나 일부 사례는 특정 산업의 특수성을 반영하지 못해 약간의 차이가 있다.

2) **확장성**: 4점  
   - 다양한 산업(유통, 전자상거래, 패션 등)에서 수요 예측을 다루고 있어 표현이 달라져도 자연스럽게 확장 가능하다. 그러나 특정 산업에 국한된 사례가 있어 범위가 다소 제한적이다.

3) **근거 품질**: 4점  
   - 출처 URL이 명확하고 신뢰할 수 있는 정보로 연결되어 있으며, 각 사례에 대한 구체적인 디테일이 잘 설명되어 있다. 다만, 출처의 다양성이 조금 부족하다.

4) **중복 최소화**: 4점  
   - 각 사례가 서로 다른 기업을 다루고 있어 중복이 적고, 사례 간 구분이 명확하다. 그러나 유사한 산업 내에서의 반복성이 다소 느껴질 수 있다.

**총점**: 16점

---

### EXA (Semantic Search) 평가

1) **유사성 적합도**: 5점  
   - Seed Case와 모든 사례가 과거 데이터와 고객 데이터를 활용하여 수요 예측을 수행하는 점에서 매우 유사하다. 각 사례는 명확하게 문제와 목표를 설정하고 있다.

2) **확장성**: 5점  
   - 다양한 산업(식품, 제약, 물류 등)에서 수요 예측을 다루고 있어 표현이 달라져도 자연스럽게 확장 가능하다. 각 사례가 서로 다른 맥락을 잘 반영하고 있다.

3) **근거 품질**: 5점  
   - 출처 URL이 신뢰할 수 있는 정보로 연결되어 있으며, 각 사례에 대한 구체적인 디테일이 잘 설명되어 있다. 출처의 다양성도 높다.

4) **중복 최소화**: 5점  
   - 각 사례가 서로 다른 기업을 다루고 있어 중복이 없고, 사례 간 구분이 명확하다. 다양

### 1️⃣3️⃣ 결과 저장

In [26]:
from datetime import datetime

os.makedirs("outputs", exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"outputs/chapter3_research_{timestamp}.md"

with open(filename, "w", encoding="utf-8") as f:
    f.write("# 실습 3 - 유사 사례 확장 리서치\n\n")
    f.write(f"**Seed Case:** {seed_case}\n\n")
    f.write("## SERPER 결과\n\n")
    f.write(str(serper_result))
    f.write("\n\n## EXA 결과\n\n")
    f.write(str(exa_result))
    f.write("\n\n## LLM 평가\n\n")
    f.write(resp.choices[0].message.content)

print(f"\n✅ 결과가 저장되었습니다:\n{filename}")


✅ 결과가 저장되었습니다:
outputs/chapter3_research_20260525_173342.md


## 🔎 결과를 관찰해 보세요

1. SERPER와 EXA 중 어느 쪽이 더 유사성 높은 사례를 찾았는가?
2. 표현이 달라도 연결되는 사례가 있었는가? 어느 도구에서 더 많이 나왔는가?
3. 새로운 산업/맥락으로 자연스럽게 확장된 사례가 있었는가?
4. 출처 신뢰도 측면에서 두 도구의 차이가 있었는가?
5. 2장(SERPER만 사용)과 비교했을 때 EXA가 추가로 찾아낸 사례의 특징은 무엇인가?

👉 키워드 검색은 정확한 단어에 강하고, 의미 기반 검색은 다양한 표현으로 분산된 사례 탐색에 강합니다.